In [23]:
import json
from pathlib import Path

import numpy as np


def find_project_root() -> Path:
    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for path in search_roots:
        if (
            (path / 'embeddings' / 'product_embeddings.npy').exists()
            and (path / 'embeddings' / 'embedding_metadata.json').exists()
        ):
            return path
    raise FileNotFoundError(
        'Could not locate the project root. Run the notebook from the repo or one of its subfolders.'
    )


project_root = find_project_root()
embeddings = np.load(project_root / 'embeddings' / 'product_embeddings.npy')
metadata = json.loads((project_root / 'embeddings' / 'embedding_metadata.json').read_text(encoding='utf-8'))
evaluation = json.loads((project_root / 'embeddings' / 'embedding_evaluation.json').read_text(encoding='utf-8'))

print('Project root:', project_root)
print('Embedding shape:', embeddings.shape)
print('Product count:', metadata['product_count'])
print('Model:', metadata['model_name'])
print('First three products:', [product['title'] for product in metadata['products'][:3]])

Project root: C:\Users\spune\Desktop\GenAI_Recommender_DyanamoB_Sprint2_Week3
Embedding shape: (500, 384)
Product count: 500
Model: all-MiniLM-L6-v2
First three products: ['Itself Sofa', 'There Sofa', 'Task Wardrobe']


In [24]:
products = metadata['products']
similarity_matrix = embeddings @ embeddings.T
upper_triangle = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

print('Similarity min:', round(float(upper_triangle.min()), 4))
print('Similarity max:', round(float(upper_triangle.max()), 4))
print('Similarity mean:', round(float(upper_triangle.mean()), 4))
print('Same-category similarity mean:', evaluation['similarity_score_distribution']['same_category_pairs']['mean'])
print('Different-category similarity mean:', evaluation['similarity_score_distribution']['different_category_pairs']['mean'])
print('Precision@5:', evaluation['retrieval_metrics']['precision_at_5'])
print('Recall@10:', evaluation['retrieval_metrics']['recall_at_10'])
print('NDCG@10:', evaluation['retrieval_metrics']['ndcg_at_10'])

top_pair_indices = np.dstack(np.unravel_index(np.argsort(similarity_matrix, axis=None)[::-1], similarity_matrix.shape))[0]
seen_pairs = set()
top_pairs = []
for left_idx, right_idx in top_pair_indices:
    if left_idx == right_idx:
        continue
    pair_key = tuple(sorted((int(left_idx), int(right_idx))))
    if pair_key in seen_pairs:
        continue
    seen_pairs.add(pair_key)
    top_pairs.append((float(similarity_matrix[left_idx, right_idx]), products[left_idx]['title'], products[right_idx]['title']))
    if len(top_pairs) == 5:
        break

for score, left_title, right_title in top_pairs:
    print(f'{left_title} <-> {right_title}: {score:.4f}')

Similarity min: 0.2513
Similarity max: 1.0
Similarity mean: 0.6743
Same-category similarity mean: 0.8353
Different-category similarity mean: 0.6343
Precision@5: 1.0
Recall@10: 0.1285
NDCG@10: 0.993
Skin Chair <-> Join Chair: 1.0000
Part Bed <-> Sort Bed: 0.9969
Front Bed <-> Song Bed: 0.9963
Property Chair <-> Present Chair: 0.9963
Open Bed <-> Charge Bed: 0.9956


In [25]:
import importlib
import sys
from pathlib import Path

src_path = project_root / 'src'
sys.path = [str(src_path), *[entry for entry in sys.path if entry != str(src_path)]]

importlib.invalidate_caches()
for module_name in ('embedder', 'vector_store', 'content_recommender'):
    sys.modules.pop(module_name, None)

embedder = importlib.import_module('embedder')
print('Loaded embedder from:', Path(embedder.__file__).resolve())
print('Embedder model key:', embedder.DEFAULT_MODEL_KEY)

vector_store = importlib.import_module('vector_store')
print('Loaded vector_store from:', Path(vector_store.__file__).resolve())

search_by_text = vector_store.search_by_text

queries = [
    {'text': 'modern fabric sofa', 'filters': {'category': 'Sofa'}},
    {'text': 'industrial wood wardrobe', 'filters': {'category': 'Wardrobe'}},
    {'text': 'minimalist grey chair', 'filters': {'category': 'Chair', 'min_price': 100, 'max_price': 1200}},
]

for item in queries:
    print('Query:', item['text'])
    print('Filters:', item['filters'])
    results = search_by_text(
        item['text'],
        top_k=3,
        category=item['filters'].get('category'),
        min_price=item['filters'].get('min_price'),
        max_price=item['filters'].get('max_price'),
    )
    for result in results:
        print(f"  {result['rank']}. {result['title']} ({result['product_id']}) | score={result['score']:.4f} | {result['description']}")
    print()


Loaded embedder from: C:\Users\spune\Desktop\GenAI_Recommender_DyanamoB_Sprint2_Week3\src\embedder.py
Embedder model key: all-MiniLM-L6-v2
Loaded vector_store from: C:\Users\spune\Desktop\GenAI_Recommender_DyanamoB_Sprint2_Week3\src\vector_store.py
Query: modern fabric sofa
Filters: {'category': 'Sofa'}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  1. Position Sofa (349) | score=0.7788 | Modern Fabric Sofa in White color. Price: 872.
  2. Machine Sofa (498) | score=0.7760 | Modern Fabric Sofa in White color. Price: 564.
  3. Your Sofa (278) | score=0.7695 | Modern Fabric Sofa in Black color. Price: 886.

Query: industrial wood wardrobe
Filters: {'category': 'Wardrobe'}
  1. Film Wardrobe (135) | score=0.7524 | Industrial Wood Wardrobe in Black color. Price: 331.
  2. Task Wardrobe (3) | score=0.7506 | Industrial Wood Wardrobe in White color. Price: 306.
  3. Chance Wardrobe (447) | score=0.7442 | Industrial Wood Wardrobe in Brown color. Price: 817.

Query: minimalist grey chair
Filters: {'category': 'Chair', 'min_price': 100, 'max_price': 1200}
  1. Agree Chair (188) | score=0.8177 | Minimalist Metal Chair in Grey color. Price: 141.
  2. Fight Chair (380) | score=0.8176 | Minimalist Metal Chair in Grey color. Price: 107.
  3. They Chair (4) | score=0.7991 | Minimalist Wood Chair in Grey color. Price: 136.

